# HT-Demucs and Band-Split RoFormer Training Pipeline
## Complete Implementation for Audio Source Separation

This notebook implements the complete training pipeline for two state-of-the-art source separation models:
1. **HT-Demucs** (Hybrid Transformer Demucs) - Waveform-based approach
2. **Band-Split RoFormer** - Spectrogram-based with frequency band processing

Both models will be trained on the preprocessed MUSDB18 dataset.

## Phase 1: Import Dependencies and Setup

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchaudio
import torchaudio.transforms as T
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
GPU Memory: 6.44 GB


In [2]:
# Configuration
CONFIG = {
    'data_dir': './musdb18_processed',  
    'checkpoint_dir': '../checkpoints',
    'log_dir': '../logs',
    'sample_rate': 44100,
    'n_fft': 4096,
    'hop_length': 1024,
    'num_sources': 4,  # vocals, drums, bass, other
    'source_names': ['vocals', 'drums', 'bass', 'other'],
    'random_seed': 42,
    'train_val_split': 0.8,
}

# Create directories
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['log_dir'], exist_ok=True)

# Set random seeds for reproducibility
torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['random_seed'])

print("Configuration loaded successfully!")

Configuration loaded successfully!


## Phase 2: Data Loading Infrastructure

In [3]:
class MUSDB18Dataset(Dataset):
    """
    Dataset loader for preprocessed MUSDB18 .npz files.
    Supports both HT-Demucs (waveform) and BSRoFormer (spectrogram) modes.
    """
    def __init__(self, data_dir, subset='train', model_type='htdemucs', 
                 augment=False, cache_data=False):
        """
        Args:
            data_dir: Path to musdb18_processed directory
            subset: 'train' or 'test'
            model_type: 'htdemucs' (waveform) or 'bsroformer' (spectrogram)
            augment: Apply data augmentation (training only)
            cache_data: Load all data into memory (requires 50GB+ RAM)
        """
        self.data_dir = Path(data_dir) / subset
        self.model_type = model_type
        self.augment = augment and subset == 'train'
        self.cache_data = cache_data
        
        # Find all .npz files
        self.file_paths = sorted(list(self.data_dir.glob('*.npz')))
        print(f"Found {len(self.file_paths)} files in {subset} set")
        
        # Cache data if requested
        self.cache = {}
        if cache_data:
            print("Caching data in memory...")
            for idx, path in enumerate(tqdm(self.file_paths)):
                self.cache[idx] = np.load(path)
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        # Load data
        if self.cache_data and idx in self.cache:
            data = self.cache[idx]
        else:
            data = np.load(self.file_paths[idx])
        
        if self.model_type == 'htdemucs':
            return self._get_waveform_data(data)
        else:  # bsroformer
            return self._get_spectrogram_data(data)
    
    def _get_waveform_data(self, data):
        """Extract waveform data for HT-Demucs"""
        # Load mixture and sources - shape is (time, channels), need (channels, time)
        mixture = torch.from_numpy(data['mixture_wav']).float().transpose(0, 1)  # (2, time)
        sources = torch.stack([
            torch.from_numpy(data['source_0']).float().transpose(0, 1),  # vocals
            torch.from_numpy(data['source_1']).float().transpose(0, 1),  # drums
            torch.from_numpy(data['source_2']).float().transpose(0, 1),  # bass
            torch.from_numpy(data['source_3']).float().transpose(0, 1),  # other
        ])  # (4, 2, time)
        
        # Apply augmentations if training
        if self.augment:
            mixture, sources = self._augment_waveform(mixture, sources)
        
        return mixture, sources
    
    def _get_spectrogram_data(self, data):
        """Extract spectrogram data for BSRoFormer"""
        # Load magnitude/log_mag and masks
        if 'log_mag' in data:
            magnitude = torch.from_numpy(data['log_mag']).float()
        else:
            magnitude = torch.from_numpy(data['mag']).float()
        
        # Load masks
        masks = torch.stack([
            torch.from_numpy(data['irm_mask_0']).float(),
            torch.from_numpy(data['irm_mask_1']).float(),
            torch.from_numpy(data['irm_mask_2']).float(),
            torch.from_numpy(data['irm_mask_3']).float(),
        ])
        
        # Load phase for reconstruction
        phase = torch.from_numpy(data['phase']).float()
        
        # Load waveforms for reconstruction loss - transpose to (channels, time)
        mixture_wav = torch.from_numpy(data['mixture_wav']).float().transpose(0, 1)  # (2, time)
        sources_wav = torch.stack([
            torch.from_numpy(data['source_0']).float().transpose(0, 1),  # vocals
            torch.from_numpy(data['source_1']).float().transpose(0, 1),  # drums
            torch.from_numpy(data['source_2']).float().transpose(0, 1),  # bass
            torch.from_numpy(data['source_3']).float().transpose(0, 1),  # other
        ])  # (4, 2, time)
        
        # Apply augmentations if training
        if self.augment:
            magnitude, masks = self._augment_spectrogram(magnitude, masks)
        
        return magnitude, masks, phase, mixture_wav, sources_wav
    
    def _augment_waveform(self, mixture, sources):
        """Apply waveform augmentations for HT-Demucs"""
        # Random gain adjustment (±6 dB)
        if np.random.rand() < 0.5:
            gain_db = np.random.uniform(-6, 6)
            gain = 10 ** (gain_db / 20)
            mixture = mixture * gain
            sources = sources * gain
        
        # Channel swapping (for stereo data)
        if mixture.shape[0] == 2 and np.random.rand() < 0.3:
            mixture = mixture.flip(0)
            sources = sources.flip(1)
        
        # Source remixing (random linear combinations)
        if np.random.rand() < 0.3:
            remix_weights = torch.rand(4) * 0.2 + 0.9  # [0.9, 1.1]
            sources = sources * remix_weights[:, None, None]
            mixture = sources.sum(dim=0)
        
        return mixture, sources
    
    def _augment_spectrogram(self, magnitude, masks):
        """Apply spectrogram augmentations for BSRoFormer"""
        # SpecAugment: frequency masking
        if np.random.rand() < 0.5:
            freq_mask_param = int(magnitude.shape[-2] * 0.1)
            freq_start = np.random.randint(0, magnitude.shape[-2] - freq_mask_param)
            magnitude[..., freq_start:freq_start+freq_mask_param, :] *= 0.0
        
        # SpecAugment: time masking
        if np.random.rand() < 0.5:
            time_mask_param = int(magnitude.shape[-1] * 0.1)
            time_start = np.random.randint(0, magnitude.shape[-1] - time_mask_param)
            magnitude[..., time_start:time_start+time_mask_param] *= 0.0
        
        # Random gain in spectral domain
        if np.random.rand() < 0.5:
            gain = torch.rand(1) * 0.4 + 0.8  # [0.8, 1.2]
            magnitude = magnitude * gain
        
        return magnitude, masks

print("MUSDB18Dataset class defined successfully!")

MUSDB18Dataset class defined successfully!


In [4]:
def create_dataloaders(data_dir, model_type='htdemucs', batch_size=4, 
                       num_workers=0, train_val_split=0.8, augment=True):
    """
    Create train and validation dataloaders.
    
    Args:
        data_dir: Path to musdb18_processed directory
        model_type: 'htdemucs' or 'bsroformer'
        batch_size: Batch size
        num_workers: Number of data loading workers (set to 0 for Windows to avoid multiprocessing issues)
        train_val_split: Fraction of training data for training (rest for validation)
        augment: Apply data augmentation to training set
    
    Returns:
        train_loader, val_loader
    """
    # Load full training dataset
    full_train_dataset = MUSDB18Dataset(
        data_dir, subset='train', model_type=model_type, 
        augment=augment, cache_data=False
    )
    
    # Split into train and validation
    train_size = int(train_val_split * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(CONFIG['random_seed'])
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True, drop_last=False
    )
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Training batches: {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")
    
    return train_loader, val_loader

# Test data loading
print("DataLoader creation function defined successfully!")

DataLoader creation function defined successfully!


## Phase 3: HT-Demucs Architecture and Training

In [5]:
class HTDemucs(nn.Module):
    """
    Hybrid Transformer Demucs architecture for source separation.
    Combines time-domain and frequency-domain processing with Transformer.
    """
    def __init__(self, channels=48, depth=6, kernel_size=8, stride=4,
                 num_transformer_layers=5, num_heads=8, d_model=384,
                 num_sources=4, dropout=0.1):
        """
        Args:
            channels: Base number of channels
            depth: Number of encoder/decoder layers
            kernel_size: Convolution kernel size
            stride: Downsampling factor
            num_transformer_layers: Number of Transformer layers
            num_heads: Number of attention heads
            d_model: Transformer dimension
            num_sources: Number of sources to separate
            dropout: Dropout rate
        """
        super().__init__()
        self.channels = channels
        self.depth = depth
        self.num_sources = num_sources
        
        # Track encoder output channels (after GLU)
        self.encoder_channels = []
        
        # Time-domain encoder (U-Net style with GLU)
        self.encoder = nn.ModuleList()
        in_ch = 2  # stereo input
        for i in range(depth):
            out_ch = channels * (2 ** i)
            self.encoder.append(nn.Sequential(
                nn.Conv1d(in_ch, out_ch * 2, kernel_size, stride, padding=kernel_size//2),
                nn.BatchNorm1d(out_ch * 2),
                nn.GLU(dim=1)  # GLU halves channels: out_ch*2 -> out_ch
            ))
            self.encoder_channels.append(out_ch)
            in_ch = out_ch
        
        # Transformer in frequency domain
        final_ch = self.encoder_channels[-1]
        self.lstm = nn.LSTM(
            final_ch, d_model // 2, num_transformer_layers,
            batch_first=True, bidirectional=True, dropout=dropout
        )
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=num_heads, 
                dim_feedforward=d_model*4, dropout=dropout,
                batch_first=True
            ),
            num_layers=num_transformer_layers
        )
        
        # Project from transformer back to channels
        self.transformer_proj = nn.Linear(d_model, final_ch)
        
        # Time-domain decoder (mirror of encoder)
        self.decoder = nn.ModuleList()
        for i in range(depth-1, -1, -1):
            # Input: current features + skip connection
            in_ch = self.encoder_channels[i] * 2  # Concatenated with skip
            # Output: previous encoder layer's channels (or final output)
            out_ch = self.encoder_channels[i-1] if i > 0 else channels
            
            self.decoder.append(nn.Sequential(
                nn.ConvTranspose1d(in_ch, out_ch * 2, kernel_size, stride, 
                                   padding=kernel_size//2, output_padding=stride-1),
                nn.BatchNorm1d(out_ch * 2),
                nn.GLU(dim=1)  # out_ch*2 -> out_ch
            ))
        
        # Final layer to produce sources
        self.final = nn.Conv1d(channels, num_sources * 2, 1)
    
    def forward(self, x):
        """
        Args:
            x: [batch, 2, time] - stereo mixture waveform
        Returns:
            [batch, num_sources, 2, time] - separated sources
        """
        batch = x.shape[0]
        
        # Encoder
        skip_connections = []
        for layer in self.encoder:
            x = layer(x)
            skip_connections.append(x)
        
        # Reshape for Transformer: [batch, channels, time] -> [batch, time, channels]
        x = x.transpose(1, 2)
        
        # LSTM + Transformer
        x, _ = self.lstm(x)
        x = self.transformer(x)
        
        # Project back to channel dimension
        x = self.transformer_proj(x)
        
        # Reshape back: [batch, time, channels] -> [batch, channels, time]
        x = x.transpose(1, 2)
        
        # Decoder with skip connections
        for i, layer in enumerate(self.decoder):
            # Get corresponding skip connection
            skip = skip_connections[-(i+1)]
            
            # Match time dimension if needed
            if x.shape[-1] != skip.shape[-1]:
                x = F.interpolate(x, size=skip.shape[-1], mode='linear', align_corners=False)
            
            # Concatenate with skip connection
            x = torch.cat([x, skip], dim=1)
            
            # Apply decoder layer
            x = layer(x)
        
        # Final prediction
        x = self.final(x)
        
        # Reshape to [batch, num_sources, 2, time]
        x = x.view(batch, self.num_sources, 2, -1)
        
        return x

print("HTDemucs model architecture defined!")

HTDemucs model architecture defined!


In [6]:
class HTDemucsLoss(nn.Module):
    """
    Combined loss for HT-Demucs: time-domain + multi-scale STFT loss.
    """
    def __init__(self, stft_scales=[2048, 1024, 512, 256, 128, 64],
                 time_loss_weight=1.0, freq_loss_weight=1.0):
        super().__init__()
        self.stft_scales = stft_scales
        self.time_loss_weight = time_loss_weight
        self.freq_loss_weight = freq_loss_weight
        
        # Create STFT transforms for each scale
        self.stfts = nn.ModuleList([
            T.Spectrogram(n_fft=n_fft, hop_length=n_fft//4, 
                         power=None, return_complex=True)
            for n_fft in stft_scales
        ])
    
    def time_domain_loss(self, pred, target):
        """L1 loss in time domain"""
        return F.l1_loss(pred, target)
    
    def stft_loss(self, pred, target):
        """Multi-scale STFT loss"""
        total_loss = 0.0
        
        # Flatten to [batch*sources, channels, time]
        pred_flat = pred.reshape(-1, pred.shape[-2], pred.shape[-1])
        target_flat = target.reshape(-1, target.shape[-2], target.shape[-1])
        
        for stft in self.stfts:
            # Compute STFT
            pred_stft = stft(pred_flat)
            target_stft = stft(target_flat)
            
            # Linear magnitude loss
            pred_mag = pred_stft.abs()
            target_mag = target_stft.abs()
            linear_loss = F.l1_loss(pred_mag, target_mag)
            
            # Log magnitude loss (better for perceptual quality)
            pred_log = torch.log1p(pred_mag)
            target_log = torch.log1p(target_mag)
            log_loss = F.l1_loss(pred_log, target_log)
            
            total_loss += (linear_loss + log_loss) / len(self.stfts)
        
        return total_loss
    
    def forward(self, pred_sources, target_sources):
        """
        Args:
            pred_sources: [batch, num_sources, 2, time]
            target_sources: [batch, num_sources, 2, time]
        Returns:
            loss, loss_dict with components
        """
        # Match time dimensions (pred might be slightly longer due to conv operations)
        min_time = min(pred_sources.shape[-1], target_sources.shape[-1])
        pred_sources = pred_sources[..., :min_time]
        target_sources = target_sources[..., :min_time]
        
        # Time-domain loss
        time_loss = self.time_domain_loss(pred_sources, target_sources)
        
        # Frequency-domain loss
        freq_loss = self.stft_loss(pred_sources, target_sources)
        
        # Combined loss
        total_loss = (self.time_loss_weight * time_loss + 
                     self.freq_loss_weight * freq_loss)
        
        loss_dict = {
            'total': total_loss.item(),
            'time': time_loss.item(),
            'freq': freq_loss.item()
        }
        
        return total_loss, loss_dict

print("HTDemucsLoss defined!")

HTDemucsLoss defined!


In [7]:
class ExponentialMovingAverage:
    """
    Exponential Moving Average of model weights.
    Improves model generalization and stability.
    """
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        
        # Initialize shadow weights
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self):
        """Update EMA weights"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (self.decay * self.shadow[name] + 
                                    (1 - self.decay) * param.data)
    
    def apply_shadow(self):
        """Apply EMA weights to model"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
    
    def restore(self):
        """Restore original weights"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

print("EMA helper class defined!")

EMA helper class defined!


In [8]:
class HTDemucsTrainer:
    """
    Training pipeline for HT-Demucs model.
    """
    def __init__(self, model, train_loader, val_loader, config):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        
        # Loss function
        self.criterion = HTDemucsLoss().to(device)
        
        # Optimizer with weight decay
        self.optimizer = AdamW(
            model.parameters(),
            lr=config.get('learning_rate', 3e-4),
            weight_decay=config.get('weight_decay', 1e-5),
            betas=(0.9, 0.999)
        )
        
        # Learning rate scheduler (cosine annealing with warmup)
        warmup_epochs = config.get('warmup_epochs', 10)
        total_epochs = config.get('num_epochs', 360)
        self.warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            self.optimizer, start_factor=0.1, total_iters=warmup_epochs
        )
        self.cosine_scheduler = CosineAnnealingLR(
            self.optimizer, T_max=total_epochs-warmup_epochs, eta_min=1e-7
        )
        self.current_epoch = 0
        
        # EMA for model weights
        self.ema = ExponentialMovingAverage(
            model, decay=config.get('ema_decay', 0.999)
        )
        
        # Gradient clipping
        self.grad_clip = config.get('grad_clip', 5.0)
        
        # Mixed precision training
        self.use_amp = config.get('use_amp', True)
        self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None
        
        # Tracking
        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float('inf')
        self.checkpoint_dir = Path(config.get('checkpoint_dir', '../checkpoints/htdemucs'))
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        epoch_losses = []
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {self.current_epoch}")
        for batch_idx, (mixture, sources) in enumerate(pbar):
            mixture = mixture.to(device)
            sources = sources.to(device)
            
            # Forward pass with mixed precision
            self.optimizer.zero_grad()
            
            if self.use_amp:
                with torch.cuda.amp.autocast():
                    pred_sources = self.model(mixture)
                    loss, loss_dict = self.criterion(pred_sources, sources)
                
                # Backward pass
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                pred_sources = self.model(mixture)
                loss, loss_dict = self.criterion(pred_sources, sources)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.optimizer.step()
            
            # Update EMA
            self.ema.update()
            
            epoch_losses.append(loss_dict)
            pbar.set_postfix({
                'loss': f"{loss_dict['total']:.4f}",
                'time': f"{loss_dict['time']:.4f}",
                'freq': f"{loss_dict['freq']:.4f}"
            })
        
        # Average losses
        avg_loss = {
            key: np.mean([d[key] for d in epoch_losses])
            for key in epoch_losses[0].keys()
        }
        
        return avg_loss
    
    def validate(self):
        """Validate on validation set"""
        self.model.eval()
        val_losses = []
        
        with torch.no_grad():
            for mixture, sources in tqdm(self.val_loader, desc="Validation"):
                mixture = mixture.to(device)
                sources = sources.to(device)
                
                if self.use_amp:
                    with torch.cuda.amp.autocast():
                        pred_sources = self.model(mixture)
                        loss, loss_dict = self.criterion(pred_sources, sources)
                else:
                    pred_sources = self.model(mixture)
                    loss, loss_dict = self.criterion(pred_sources, sources)
                
                val_losses.append(loss_dict)
        
        # Average losses
        avg_loss = {
            key: np.mean([d[key] for d in val_losses])
            for key in val_losses[0].keys()
        }
        
        return avg_loss
    
    def save_checkpoint(self, is_best=False, epoch=None):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': self.current_epoch,
            'model_state_dict': self.model.state_dict(),
            'ema_shadow': self.ema.shadow,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'warmup_scheduler_state_dict': self.warmup_scheduler.state_dict(),
            'cosine_scheduler_state_dict': self.cosine_scheduler.state_dict(),
            'best_val_loss': self.best_val_loss,
            'config': self.config
        }
        
        # Save regular checkpoint
        if epoch is not None:
            path = self.checkpoint_dir / f'htdemucs_epoch_{epoch:03d}.pt'
            torch.save(checkpoint, path)
            print(f"Saved checkpoint: {path}")
        
        # Save best checkpoint
        if is_best:
            path = self.checkpoint_dir / 'htdemucs_best.pt'
            torch.save(checkpoint, path)
            print(f"Saved best checkpoint: {path}")
    
    def train(self, num_epochs):
        """Main training loop"""
        print(f"Starting HT-Demucs training for {num_epochs} epochs...")
        
        for epoch in range(num_epochs):
            self.current_epoch = epoch + 1
            
            # Train
            train_loss = self.train_epoch()
            self.train_losses.append(train_loss)
            
            # Validate every 5 epochs
            if (epoch + 1) % 5 == 0:
                val_loss = self.validate()
                self.val_losses.append(val_loss)
                
                print(f"Epoch {self.current_epoch}:")
                print(f"  Train Loss: {train_loss['total']:.4f} "
                      f"(time: {train_loss['time']:.4f}, freq: {train_loss['freq']:.4f})")
                print(f"  Val Loss: {val_loss['total']:.4f} "
                      f"(time: {val_loss['time']:.4f}, freq: {val_loss['freq']:.4f})")
                
                # Save best model
                if val_loss['total'] < self.best_val_loss:
                    self.best_val_loss = val_loss['total']
                    self.save_checkpoint(is_best=True)
            
            # Save checkpoint every 10 epochs
            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(epoch=self.current_epoch)
            
            # Step scheduler
            if epoch < self.config.get('warmup_epochs', 10):
                self.warmup_scheduler.step()
            else:
                self.cosine_scheduler.step()
        
        print("Training completed!")
        self.save_checkpoint(epoch=num_epochs)

print("HTDemucsTrainer class defined!")

HTDemucsTrainer class defined!


## Phase 4: Band-Split RoFormer Architecture and Training

In [9]:
class RotaryPositionEmbedding(nn.Module):
    """
    Rotary Position Embedding (RoPE) for Transformer.
    Better than absolute position encoding for variable-length sequences.
    """
    def __init__(self, dim, max_seq_len=5000):
        super().__init__()
        # dim should be the head_dim, and we compute for half of it
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self.max_seq_len = max_seq_len
    
    def forward(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)  # [seq_len, dim//2]
        # Don't concatenate - return as is for half the head dimension
        return freqs.cos(), freqs.sin()

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    Apply rotary position embedding to queries and keys.
    Args:
        q, k: [batch, num_heads, seq_len, head_dim]
        cos, sin: [1, 1, seq_len, head_dim//2]
    """
    # Split into two halves along head_dim
    q1, q2 = q.chunk(2, dim=-1)  # Each: [batch, num_heads, seq_len, head_dim//2]
    k1, k2 = k.chunk(2, dim=-1)
    
    # Apply rotation
    q_rot = torch.cat([
        q1 * cos - q2 * sin,
        q1 * sin + q2 * cos
    ], dim=-1)
    k_rot = torch.cat([
        k1 * cos - k2 * sin,
        k1 * sin + k2 * cos
    ], dim=-1)
    
    return q_rot, k_rot

print("Rotary Position Embedding defined!")

Rotary Position Embedding defined!


In [10]:
class RoFormerLayer(nn.Module):
    """
    Single RoFormer layer with rotary position embedding.
    """
    def __init__(self, d_model, num_heads, dim_feedforward, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Multi-head attention
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        # Feedforward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model)
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Rotary position embedding
        self.rope = RotaryPositionEmbedding(self.head_dim)
    
    def forward(self, x):
        """
        Args:
            x: [batch, seq_len, d_model]
        """
        batch, seq_len, _ = x.shape
        
        # Self-attention with RoPE
        residual = x
        x = self.norm1(x)
        
        # Project to Q, K, V
        q = self.q_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Apply rotary position embedding
        cos, sin = self.rope(seq_len)
        cos = cos[None, None, :, :].to(x.device)
        sin = sin[None, None, :, :].to(x.device)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # Attention
        attn = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        out = self.out_proj(out)
        out = self.dropout(out)
        x = residual + out
        
        # Feedforward
        residual = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x = residual + x
        
        return x

print("RoFormerLayer defined!")

RoFormerLayer defined!


In [11]:
class BandSplitRoFormer(nn.Module):
    """
    Band-Split RoFormer for source separation.
    Splits spectrum into bands, processes each with RoFormer, predicts masks.
    """
    def __init__(self, num_sources=4, d_model=384, num_layers=12, num_heads=8,
                 dim_feedforward=1536, dropout=0.1, n_fft=4096, sample_rate=44100):
        """
        Args:
            num_sources: Number of sources to separate
            d_model: Transformer dimension
            num_layers: Number of Transformer layers
            num_heads: Number of attention heads
            dim_feedforward: FFN dimension
            dropout: Dropout rate
            n_fft: FFT size (determines frequency bins)
            sample_rate: Audio sample rate
        """
        super().__init__()
        self.num_sources = num_sources
        self.d_model = d_model
        self.n_fft = n_fft
        self.sample_rate = sample_rate
        
        # Frequency band boundaries (Hz)
        # Low: 0-1500, Mid: 1500-6000, High: 6000-22050
        self.band_boundaries = [0, 1500, 6000, 22050]
        
        # Calculate frequency bins for each band
        freq_bins = n_fft // 2 + 1  # Number of frequency bins
        freq_resolution = sample_rate / 2 / freq_bins
        
        self.band_freq_bins = []
        for i in range(len(self.band_boundaries) - 1):
            low_hz = self.band_boundaries[i]
            high_hz = self.band_boundaries[i + 1]
            low_bin = int(low_hz / freq_resolution)
            high_bin = int(high_hz / freq_resolution)
            self.band_freq_bins.append(high_bin - low_bin)
        
        # Input projection for each band (freq_bins -> d_model)
        self.band_projections = nn.ModuleList([
            nn.Linear(freq_bins, d_model) for freq_bins in self.band_freq_bins
        ])
        
        # Transformer for each band
        self.band_transformers = nn.ModuleList([
            nn.Sequential(*[
                RoFormerLayer(d_model, num_heads, dim_feedforward, dropout)
                for _ in range(num_layers)
            ]) for _ in range(3)
        ])
        
        # Mask prediction heads (d_model -> num_sources, then upsample back to freq_bins)
        self.mask_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_model // 2),
                nn.ReLU(),
                nn.Linear(d_model // 2, num_sources),
                nn.Sigmoid()  # Masks in [0, 1]
            ) for _ in range(3)
        ])
    
    def split_bands(self, spec):
        """
        Split spectrogram into frequency bands.
        Args:
            spec: [batch, freq, time] magnitude spectrogram
        Returns:
            List of 3 band spectrograms
        """
        freq_bins = spec.shape[1]
        freq_resolution = self.sample_rate / 2 / freq_bins
        
        bands = []
        for i in range(len(self.band_boundaries) - 1):
            low_hz = self.band_boundaries[i]
            high_hz = self.band_boundaries[i + 1]
            
            low_bin = int(low_hz / freq_resolution)
            high_bin = int(high_hz / freq_resolution)
            
            band = spec[:, low_bin:high_bin, :]
            bands.append(band)
        
        return bands
    
    def forward(self, magnitude):
        """
        Args:
            magnitude: [batch, freq, time] magnitude spectrogram
        Returns:
            masks: [batch, num_sources, freq, time]
        """
        batch, freq, time = magnitude.shape
        
        # Split into bands
        bands = self.split_bands(magnitude)
        
        # Process each band
        band_masks = []
        for band_idx, band in enumerate(bands):
            band_freq = band.shape[1]  # Frequency bins in this band
            
            # Reshape: [batch, freq, time] -> [batch, time, freq]
            band = band.transpose(1, 2)
            
            # Project to d_model: [batch, time, freq] -> [batch, time, d_model]
            band = self.band_projections[band_idx](band)
            
            # Apply Transformer: [batch, time, d_model] -> [batch, time, d_model]
            band = self.band_transformers[band_idx](band)
            
            # Predict masks: [batch, time, d_model] -> [batch, time, num_sources]
            masks = self.mask_heads[band_idx](band)
            
            # Reshape: [batch, time, num_sources] -> [batch, num_sources, time]
            masks = masks.transpose(1, 2)
            
            # Expand to frequency dimension: [batch, num_sources, time] -> [batch, num_sources, freq, time]
            # Each time step gets the same mask across all frequencies in this band
            masks = masks.unsqueeze(2).expand(-1, -1, band_freq, -1)
            
            band_masks.append(masks)
        
        # Concatenate bands along frequency dimension: [batch, num_sources, freq, time]
        masks = torch.cat(band_masks, dim=2)
        
        # Ensure masks match input frequency dimension (handle rounding issues)
        if masks.shape[2] != freq:
            masks = F.interpolate(masks, size=(freq, time), mode='bilinear', align_corners=False)
        
        return masks

print("BandSplitRoFormer model defined!")

BandSplitRoFormer model defined!


In [12]:
class BSRoFormerLoss(nn.Module):
    """
    Combined loss for Band-Split RoFormer: mask loss + reconstruction loss.
    """
    def __init__(self, n_fft=4096, hop_length=1024, 
                 mask_loss_weight=1.0, recon_loss_weight=1.0):
        super().__init__()
        self.mask_loss_weight = mask_loss_weight
        self.recon_loss_weight = recon_loss_weight
        
        # STFT for reconstruction
        self.stft = T.Spectrogram(n_fft=n_fft, hop_length=hop_length,
                                  power=None, return_complex=True)
        self.istft = T.InverseSpectrogram(n_fft=n_fft, hop_length=hop_length)
    
    def mask_loss(self, pred_masks, target_masks):
        """L1 loss between predicted and target masks"""
        return F.l1_loss(pred_masks, target_masks)
    
    def reconstruct_sources(self, masks, mixture_stft):
        """
        Reconstruct sources by applying masks to mixture.
        Args:
            masks: [batch, num_sources, freq, time]
            mixture_stft: [batch, channels, freq, time] complex
        Returns:
            [batch, num_sources, channels, time] waveforms
        """
        batch, num_sources, freq, time_masks = masks.shape
        channels = mixture_stft.shape[1]
        time_stft = mixture_stft.shape[-1]
        
        # Match time dimensions (they can differ slightly due to STFT/model operations)
        min_time = min(time_masks, time_stft)
        masks = masks[..., :min_time]
        mixture_stft = mixture_stft[..., :min_time]
        
        # Expand masks for channels
        masks = masks.unsqueeze(2)  # [batch, sources, 1, freq, time]
        mixture_stft = mixture_stft.unsqueeze(1)  # [batch, 1, channels, freq, time]
        
        # Apply masks
        separated_stft = masks * mixture_stft  # Broadcasting
        
        # ISTFT to get waveforms
        separated_stft = separated_stft.view(batch * num_sources * channels, freq, min_time)
        waveforms = self.istft(separated_stft)
        waveforms = waveforms.view(batch, num_sources, channels, -1)
        
        return waveforms
    
    def reconstruction_loss(self, pred_sources, target_sources):
        """L1 loss in time domain after reconstruction"""
        return F.l1_loss(pred_sources, target_sources)
    
    def forward(self, pred_masks, target_masks, mixture_wav, target_sources_wav):
        """
        Args:
            pred_masks: [batch, num_sources, freq, time]
            target_masks: [batch, num_sources, freq, time]
            mixture_wav: [batch, channels, time]
            target_sources_wav: [batch, num_sources, channels, time]
        Returns:
            loss, loss_dict
        """
        # Mask prediction loss
        m_loss = self.mask_loss(pred_masks, target_masks)
        
        # Reconstruction loss
        mixture_stft = self.stft(mixture_wav)
        pred_sources = self.reconstruct_sources(pred_masks, mixture_stft)
        
        # Match time dimension
        min_time = min(pred_sources.shape[-1], target_sources_wav.shape[-1])
        pred_sources = pred_sources[..., :min_time]
        target_sources_wav = target_sources_wav[..., :min_time]
        
        r_loss = self.reconstruction_loss(pred_sources, target_sources_wav)
        
        # Combined loss
        total_loss = (self.mask_loss_weight * m_loss + 
                     self.recon_loss_weight * r_loss)
        
        loss_dict = {
            'total': total_loss.item(),
            'mask': m_loss.item(),
            'recon': r_loss.item()
        }
        
        return total_loss, loss_dict

print("BSRoFormerLoss defined!")

BSRoFormerLoss defined!


In [13]:
class BSRoFormerTrainer:
    """
    Training pipeline for Band-Split RoFormer model.
    """
    def __init__(self, model, train_loader, val_loader, config):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        
        # Loss function
        self.criterion = BSRoFormerLoss(
            n_fft=config.get('n_fft', 4096),
            hop_length=config.get('hop_length', 1024)
        ).to(device)
        
        # Optimizer
        self.optimizer = AdamW(
            model.parameters(),
            lr=config.get('learning_rate', 1e-4),
            weight_decay=config.get('weight_decay', 1e-4),
            betas=(0.9, 0.98)
        )
        
        # Learning rate scheduler (warmup + cosine)
        warmup_steps = config.get('warmup_steps', 4000)
        total_steps = config.get('num_epochs', 300) * len(train_loader)
        
        self.warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            self.optimizer, start_factor=0.1, total_iters=warmup_steps
        )
        self.cosine_scheduler = CosineAnnealingLR(
            self.optimizer, T_max=total_steps-warmup_steps, eta_min=1e-6
        )
        self.current_step = 0
        self.current_epoch = 0
        
        # Gradient clipping
        self.grad_clip = config.get('grad_clip', 1.0)
        
        # Mixed precision
        self.use_amp = config.get('use_amp', True)
        self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None
        
        # Tracking
        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float('inf')
        self.checkpoint_dir = Path(config.get('checkpoint_dir', '../checkpoints/bsroformer'))
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        epoch_losses = []
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {self.current_epoch}")
        for batch_idx, (magnitude, masks, phase, mixture_wav, sources_wav) in enumerate(pbar):
            magnitude = magnitude.to(device)
            masks = masks.to(device)
            mixture_wav = mixture_wav.to(device)
            sources_wav = sources_wav.to(device)
            
            # Forward pass
            self.optimizer.zero_grad()
            
            if self.use_amp:
                with torch.cuda.amp.autocast():
                    pred_masks = self.model(magnitude)
                    loss, loss_dict = self.criterion(
                        pred_masks, masks, mixture_wav, sources_wav
                    )
                
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                pred_masks = self.model(magnitude)
                loss, loss_dict = self.criterion(
                    pred_masks, masks, mixture_wav, sources_wav
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.optimizer.step()
            
            # Step scheduler (per step, not per epoch)
            self.current_step += 1
            if self.current_step < self.config.get('warmup_steps', 4000):
                self.warmup_scheduler.step()
            else:
                self.cosine_scheduler.step()
            
            epoch_losses.append(loss_dict)
            pbar.set_postfix({
                'loss': f"{loss_dict['total']:.4f}",
                'mask': f"{loss_dict['mask']:.4f}",
                'recon': f"{loss_dict['recon']:.4f}",
                'lr': f"{self.optimizer.param_groups[0]['lr']:.2e}"
            })
        
        # Average losses
        avg_loss = {
            key: np.mean([d[key] for d in epoch_losses])
            for key in epoch_losses[0].keys()
        }
        
        return avg_loss
    
    def validate(self):
        """Validate on validation set"""
        self.model.eval()
        val_losses = []
        
        with torch.no_grad():
            for magnitude, masks, phase, mixture_wav, sources_wav in tqdm(self.val_loader, desc="Validation"):
                magnitude = magnitude.to(device)
                masks = masks.to(device)
                mixture_wav = mixture_wav.to(device)
                sources_wav = sources_wav.to(device)
                
                if self.use_amp:
                    with torch.cuda.amp.autocast():
                        pred_masks = self.model(magnitude)
                        loss, loss_dict = self.criterion(
                            pred_masks, masks, mixture_wav, sources_wav
                        )
                else:
                    pred_masks = self.model(magnitude)
                    loss, loss_dict = self.criterion(
                        pred_masks, masks, mixture_wav, sources_wav
                    )
                
                val_losses.append(loss_dict)
        
        avg_loss = {
            key: np.mean([d[key] for d in val_losses])
            for key in val_losses[0].keys()
        }
        
        return avg_loss
    
    def save_checkpoint(self, is_best=False, epoch=None):
        """Save checkpoint"""
        checkpoint = {
            'epoch': self.current_epoch,
            'step': self.current_step,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_loss': self.best_val_loss,
            'config': self.config
        }
        
        if epoch is not None:
            path = self.checkpoint_dir / f'bsroformer_epoch_{epoch:03d}.pt'
            torch.save(checkpoint, path)
            print(f"Saved checkpoint: {path}")
        
        if is_best:
            path = self.checkpoint_dir / 'bsroformer_best.pt'
            torch.save(checkpoint, path)
            print(f"Saved best checkpoint: {path}")
    
    def train(self, num_epochs):
        """Main training loop"""
        print(f"Starting BSRoFormer training for {num_epochs} epochs...")
        
        for epoch in range(num_epochs):
            self.current_epoch = epoch + 1
            
            # Train
            train_loss = self.train_epoch()
            self.train_losses.append(train_loss)
            
            # Validate every 5 epochs
            if (epoch + 1) % 5 == 0:
                val_loss = self.validate()
                self.val_losses.append(val_loss)
                
                print(f"Epoch {self.current_epoch}:")
                print(f"  Train Loss: {train_loss['total']:.4f} "
                      f"(mask: {train_loss['mask']:.4f}, recon: {train_loss['recon']:.4f})")
                print(f"  Val Loss: {val_loss['total']:.4f} "
                      f"(mask: {val_loss['mask']:.4f}, recon: {val_loss['recon']:.4f})")
                
                if val_loss['total'] < self.best_val_loss:
                    self.best_val_loss = val_loss['total']
                    self.save_checkpoint(is_best=True)
            
            # Save checkpoint every 10 epochs
            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(epoch=self.current_epoch)
        
        print("Training completed!")
        self.save_checkpoint(epoch=num_epochs)

print("BSRoFormerTrainer class defined!")

BSRoFormerTrainer class defined!


## Phase 5: Hyperparameter Optimization with Optuna

In [14]:
try:
    import optuna
    from optuna.pruners import MedianPruner
    from optuna.samplers import TPESampler
    print("Optuna imported successfully!")
except ImportError:
    print("Optuna not installed. Install with: pip install optuna")
    print("Skipping hyperparameter optimization setup.")

Optuna imported successfully!


In [15]:
def objective_htdemucs(trial, data_dir, num_epochs=20):
    """
    Optuna objective function for HT-Demucs hyperparameter search.
    Trains for limited epochs and returns validation loss.
    """
    # Sample hyperparameters
    config = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [2, 4, 8]),
        'channels': trial.suggest_categorical('channels', [32, 48, 64]),
        'depth': trial.suggest_int('depth', 5, 7),
        'num_transformer_layers': trial.suggest_int('num_transformer_layers', 4, 6),
        'dropout': trial.suggest_float('dropout', 0.0, 0.2),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True),
        'num_epochs': num_epochs,
        'warmup_epochs': 2,
        'grad_clip': 5.0,
        'use_amp': True,
        'ema_decay': 0.999,
        'checkpoint_dir': f'../checkpoints/optuna_htdemucs/trial_{trial.number}'
    }
    
    try:
        # Create dataloaders
        train_loader, val_loader = create_dataloaders(
            data_dir, model_type='htdemucs', 
            batch_size=config['batch_size'],
            num_workers=0  # 0 for Windows to avoid multiprocessing issues
        )
        
        # Create model
        model = HTDemucs(
            channels=config['channels'],
            depth=config['depth'],
            num_transformer_layers=config['num_transformer_layers'],
            dropout=config['dropout']
        )
        
        # Create trainer
        trainer = HTDemucsTrainer(model, train_loader, val_loader, config)
        
        # Train for limited epochs
        for epoch in range(num_epochs):
            train_loss = trainer.train_epoch()
            
            # Validate every 5 epochs
            if (epoch + 1) % 5 == 0:
                val_loss = trainer.validate()
                
                # Report to Optuna for pruning
                trial.report(val_loss['total'], epoch)
                
                # Check if trial should be pruned
                if trial.should_prune():
                    raise optuna.TrialPruned()
        
        # Final validation
        val_loss = trainer.validate()
        return val_loss['total']
    
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return float('inf')

def optimize_htdemucs(data_dir, n_trials=50, num_epochs=20):
    """
    Run Optuna hyperparameter optimization for HT-Demucs.
    """
    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=CONFIG['random_seed']),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    
    study.optimize(
        lambda trial: objective_htdemucs(trial, data_dir, num_epochs),
        n_trials=n_trials,
        show_progress_bar=True
    )
    
    print("\n" + "="*50)
    print("HT-Demucs Optimization Results")
    print("="*50)
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best validation loss: {study.best_value:.4f}")
    print("\nBest hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    # Save results
    results_path = Path(CONFIG['log_dir']) / 'htdemucs_optuna_results.json'
    with open(results_path, 'w') as f:
        json.dump({
            'best_params': study.best_params,
            'best_value': study.best_value,
            'n_trials': len(study.trials)
        }, f, indent=2)
    
    return study.best_params

print("HT-Demucs Optuna optimization functions defined!")

HT-Demucs Optuna optimization functions defined!


In [16]:
def objective_bsroformer(trial, data_dir, num_epochs=20):
    """
    Optuna objective function for BSRoFormer hyperparameter search.
    """
    # Sample hyperparameters
    config = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 5e-4, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [4, 8, 16]),
        'd_model': trial.suggest_categorical('d_model', [256, 384, 512]),
        'num_layers': trial.suggest_categorical('num_layers', [8, 12, 16]),
        'num_heads': trial.suggest_categorical('num_heads', [4, 8, 16]),
        'dropout': trial.suggest_float('dropout', 0.0, 0.2),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        'num_epochs': num_epochs,
        'warmup_steps': 1000,
        'grad_clip': 1.0,
        'use_amp': True,
        'n_fft': 4096,
        'hop_length': 1024,
        'checkpoint_dir': f'../checkpoints/optuna_bsroformer/trial_{trial.number}'
    }
    
    try:
        # Create dataloaders
        train_loader, val_loader = create_dataloaders(
            data_dir, model_type='bsroformer',
            batch_size=config['batch_size'],
            num_workers=0  # 0 for Windows to avoid multiprocessing issues
        )
        
        # Create model
        model = BandSplitRoFormer(
            d_model=config['d_model'],
            num_layers=config['num_layers'],
            num_heads=config['num_heads'],
            dim_feedforward=config['d_model'] * 4,
            dropout=config['dropout'],
            n_fft=config['n_fft'],
            sample_rate=CONFIG['sample_rate']
        )
        
        # Create trainer
        trainer = BSRoFormerTrainer(model, train_loader, val_loader, config)
        
        # Train for limited epochs
        for epoch in range(num_epochs):
            train_loss = trainer.train_epoch()
            
            # Validate every 5 epochs
            if (epoch + 1) % 5 == 0:
                val_loss = trainer.validate()
                
                # Report to Optuna
                trial.report(val_loss['total'], epoch)
                
                if trial.should_prune():
                    raise optuna.TrialPruned()
        
        # Final validation
        val_loss = trainer.validate()
        return val_loss['total']
    
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return float('inf')

def optimize_bsroformer(data_dir, n_trials=50, num_epochs=20):
    """
    Run Optuna hyperparameter optimization for BSRoFormer.
    """
    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=CONFIG['random_seed']),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    
    study.optimize(
        lambda trial: objective_bsroformer(trial, data_dir, num_epochs),
        n_trials=n_trials,
        show_progress_bar=True
    )
    
    print("\n" + "="*50)
    print("BSRoFormer Optimization Results")
    print("="*50)
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best validation loss: {study.best_value:.4f}")
    print("\nBest hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    # Save results
    results_path = Path(CONFIG['log_dir']) / 'bsroformer_optuna_results.json'
    with open(results_path, 'w') as f:
        json.dump({
            'best_params': study.best_params,
            'best_value': study.best_value,
            'n_trials': len(study.trials)
        }, f, indent=2)
    
    return study.best_params

print("BSRoFormer Optuna optimization functions defined!")

BSRoFormer Optuna optimization functions defined!


## Phase 6: Evaluation Metrics

In [17]:
def si_sdr(estimate, reference, eps=1e-8):
    """
    Scale-Invariant Signal-to-Distortion Ratio (SI-SDR).
    Most commonly used metric in source separation.
    
    Args:
        estimate: [batch, time] or [time] predicted signal
        reference: [batch, time] or [time] ground truth signal
        eps: Small constant for numerical stability
    
    Returns:
        SI-SDR in dB (higher is better)
    """
    # Ensure same shape
    estimate = estimate.reshape(-1)
    reference = reference.reshape(-1)
    
    # Zero-mean
    estimate = estimate - estimate.mean()
    reference = reference - reference.mean()
    
    # Compute scale
    alpha = (estimate * reference).sum() / (reference * reference).sum().clamp(min=eps)
    
    # Scale-invariant target
    target = alpha * reference
    
    # Noise (distortion)
    noise = estimate - target
    
    # SI-SDR
    si_sdr_value = 10 * torch.log10(
        (target ** 2).sum().clamp(min=eps) / (noise ** 2).sum().clamp(min=eps)
    )
    
    return si_sdr_value.item()

def sdr(estimate, reference, eps=1e-8):
    """
    Signal-to-Distortion Ratio (SDR).
    Classic BSS Eval metric.
    
    Args:
        estimate: predicted signal
        reference: ground truth signal
    
    Returns:
        SDR in dB (higher is better)
    """
    estimate = estimate.reshape(-1)
    reference = reference.reshape(-1)
    
    # Energy of reference
    ref_energy = (reference ** 2).sum().clamp(min=eps)
    
    # Energy of error
    error = estimate - reference
    error_energy = (error ** 2).sum().clamp(min=eps)
    
    sdr_value = 10 * torch.log10(ref_energy / error_energy)
    
    return sdr_value.item()

def compute_metrics(pred_sources, target_sources, source_names=['vocals', 'drums', 'bass', 'other']):
    """
    Compute SI-SDR and SDR for all sources.
    
    Args:
        pred_sources: [batch, num_sources, channels, time]
        target_sources: [batch, num_sources, channels, time]
        source_names: Names of sources
    
    Returns:
        Dictionary with metrics per source
    """
    metrics = {}
    
    # Flatten to [batch*sources*channels, time]
    pred = pred_sources.reshape(-1, pred_sources.shape[-1])
    target = target_sources.reshape(-1, target_sources.shape[-1])
    
    batch_size, num_sources = pred_sources.shape[:2]
    
    for source_idx in range(num_sources):
        source_name = source_names[source_idx]
        
        # Extract this source (all batches and channels)
        pred_source = pred_sources[:, source_idx].reshape(-1, pred_sources.shape[-1])
        target_source = target_sources[:, source_idx].reshape(-1, target_sources.shape[-1])
        
        # Compute metrics for each batch/channel and average
        si_sdr_values = []
        sdr_values = []
        
        for i in range(pred_source.shape[0]):
            si_sdr_values.append(si_sdr(pred_source[i], target_source[i]))
            sdr_values.append(sdr(pred_source[i], target_source[i]))
        
        metrics[f'{source_name}_si_sdr'] = np.mean(si_sdr_values)
        metrics[f'{source_name}_sdr'] = np.mean(sdr_values)
    
    # Overall average
    metrics['avg_si_sdr'] = np.mean([metrics[f'{name}_si_sdr'] for name in source_names])
    metrics['avg_sdr'] = np.mean([metrics[f'{name}_sdr'] for name in source_names])
    
    return metrics

print("Evaluation metrics defined!")

Evaluation metrics defined!


In [18]:
def evaluate_model(model, test_loader, model_type='htdemucs', device=device):
    """
    Evaluate model on test set with full metrics.
    
    Args:
        model: Trained model
        test_loader: DataLoader for test set
        model_type: 'htdemucs' or 'bsroformer'
        device: Device to run on
    
    Returns:
        Dictionary with evaluation results
    """
    model.eval()
    all_metrics = []
    
    print(f"Evaluating {model_type} on test set...")
    
    with torch.no_grad():
        for batch_data in tqdm(test_loader, desc="Evaluating"):
            if model_type == 'htdemucs':
                mixture, target_sources = batch_data
                mixture = mixture.to(device)
                target_sources = target_sources.to(device)
                
                # Forward pass
                pred_sources = model(mixture)
                
            else:  # bsroformer
                magnitude, masks, phase, mixture_wav, target_sources = batch_data
                magnitude = magnitude.to(device)
                mixture_wav = mixture_wav.to(device)
                target_sources = target_sources.to(device)
                
                # Forward pass
                pred_masks = model(magnitude)
                
                # Reconstruct sources
                stft = T.Spectrogram(n_fft=4096, hop_length=1024, 
                                    power=None, return_complex=True).to(device)
                istft = T.InverseSpectrogram(n_fft=4096, hop_length=1024).to(device)
                
                mixture_stft = stft(mixture_wav)
                
                # Apply masks and reconstruct
                batch, num_sources, freq, time = pred_masks.shape
                channels = mixture_stft.shape[1]
                
                pred_masks = pred_masks.unsqueeze(2)  # [batch, sources, 1, freq, time]
                mixture_stft = mixture_stft.unsqueeze(1)  # [batch, 1, channels, freq, time]
                separated_stft = pred_masks * mixture_stft
                
                separated_stft = separated_stft.view(batch * num_sources * channels, freq, time)
                pred_sources = istft(separated_stft)
                pred_sources = pred_sources.view(batch, num_sources, channels, -1)
            
            # Compute metrics
            min_time = min(pred_sources.shape[-1], target_sources.shape[-1])
            pred_sources = pred_sources[..., :min_time]
            target_sources = target_sources[..., :min_time]
            
            batch_metrics = compute_metrics(pred_sources, target_sources)
            all_metrics.append(batch_metrics)
    
    # Average metrics across all batches
    final_metrics = {}
    for key in all_metrics[0].keys():
        final_metrics[key] = np.mean([m[key] for m in all_metrics])
        final_metrics[f'{key}_std'] = np.std([m[key] for m in all_metrics])
    
    # Print results
    print("\n" + "="*60)
    print(f"Evaluation Results for {model_type.upper()}")
    print("="*60)
    
    for source in CONFIG['source_names']:
        print(f"\n{source.upper()}:")
        print(f"  SI-SDR: {final_metrics[f'{source}_si_sdr']:.2f} ± {final_metrics[f'{source}_si_sdr_std']:.2f} dB")
        print(f"  SDR:    {final_metrics[f'{source}_sdr']:.2f} ± {final_metrics[f'{source}_sdr_std']:.2f} dB")
    
    print(f"\nOVERALL:")
    print(f"  Avg SI-SDR: {final_metrics['avg_si_sdr']:.2f} ± {final_metrics['avg_si_sdr_std']:.2f} dB")
    print(f"  Avg SDR:    {final_metrics['avg_sdr']:.2f} ± {final_metrics['avg_sdr_std']:.2f} dB")
    print("="*60)
    
    return final_metrics

print("Model evaluation function defined!")

Model evaluation function defined!


## Phase 7: Training Execution and Main Workflow

### Full Training with Default Hyperparameters

In [ ]:
def train_htdemucs_full(num_epochs=360, batch_size=4):
    """
    Full training of HT-Demucs with default hyperparameters.
    Expected training time: 7-10 days on A100/V100 GPU.
    """
    print("="*60)
    print("Starting Full HT-Demucs Training")
    print("="*60)
    print(f"Epochs: {num_epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Expected duration: 7-10 days")
    print("="*60 + "\n")
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(
        CONFIG['data_dir'], model_type='htdemucs',
        batch_size=batch_size, num_workers=0, augment=True  # 0 for Windows
    )
    
    # Create model with optimal hyperparameters
    model = HTDemucs(
        channels=48,
        depth=6,
        kernel_size=8,
        stride=4,
        num_transformer_layers=5,
        num_heads=8,
        d_model=384,
        dropout=0.1
    )
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
    
    # Training configuration
    config = {
        'learning_rate': 3e-4,
        'weight_decay': 1e-5,
        'num_epochs': num_epochs,
        'warmup_epochs': 10,
        'grad_clip': 5.0,
        'use_amp': True,
        'ema_decay': 0.999,
        'checkpoint_dir': '../checkpoints/htdemucs_full'
    }
    
    # Create trainer
    trainer = HTDemucsTrainer(model, train_loader, val_loader, config)
    
    # Train
    trainer.train(num_epochs=num_epochs)
    
    print("\n✓ HT-Demucs training completed!")
    
    htdemucs_trainer = train_htdemucs_full(num_epochs=360, batch_size=4)

    # Uncomment to run full training

    return trainer

Starting Full HT-Demucs Training
Epochs: 360
Batch size: 4
Expected duration: 7-10 days

Found 188 files in train set
Training samples: 150
Validation samples: 38
Training batches: 37
Validation batches: 10
Model parameters: 91.20M
Model parameters: 91.20M
Starting HT-Demucs training for 360 epochs...
Starting HT-Demucs training for 360 epochs...


Validation: 100%|██████████| 10/10 [00:03<00:00,  2.51it/s]



Epoch 5:
  Train Loss: 0.4473 (time: 0.0667, freq: 0.3806)
  Val Loss: 0.3693 (time: 0.0564, freq: 0.3129)
Saved best checkpoint: ..\checkpoints\htdemucs_full\htdemucs_best.pt
Saved best checkpoint: ..\checkpoints\htdemucs_full\htdemucs_best.pt


Validation: 100%|██████████| 10/10 [00:03<00:00,  2.58it/s]



Epoch 10:
  Train Loss: 0.3093 (time: 0.0516, freq: 0.2578)
  Val Loss: 0.2956 (time: 0.0492, freq: 0.2465)
Saved best checkpoint: ..\checkpoints\htdemucs_full\htdemucs_best.pt
Saved best checkpoint: ..\checkpoints\htdemucs_full\htdemucs_best.pt
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_010.pt
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_010.pt


...

Epoch 350:
  Train Loss: 0.1789 (time: 0.0330, freq: 0.1460)
  Val Loss: 0.2694 (time: 0.0533, freq: 0.2161)
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_350.pt
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_350.pt


Validation: 100%|██████████| 10/10 [00:03<00:00,  2.53it/s]


Epoch 355:
  Train Loss: 0.1875 (time: 0.0343, freq: 0.1532)
  Val Loss: 0.2619 (time: 0.0514, freq: 0.2105)


Validation: 100%|██████████| 10/10 [00:03<00:00,  2.57it/s]



Epoch 360:
  Train Loss: 0.1820 (time: 0.0333, freq: 0.1487)
  Val Loss: 0.2505 (time: 0.0483, freq: 0.2022)
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_360.pt
Training completed!
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_360.pt
Training completed!
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_360.pt

✓ HT-Demucs training completed!
Saved checkpoint: ..\checkpoints\htdemucs_full\htdemucs_epoch_360.pt

✓ HT-Demucs training completed!


In [ ]:
def train_bsroformer_full(num_epochs=300, batch_size=8):
    """
    Full training of BSRoFormer with default hyperparameters.
    Expected training time: 5-7 days on A100/V100 GPU.
    """
    print("="*60)
    print("Starting Full BSRoFormer Training")
    print("="*60)
    print(f"Epochs: {num_epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Expected duration: 5-7 days")
    print("="*60 + "\n")
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(
        CONFIG['data_dir'], model_type='bsroformer',
        batch_size=batch_size, num_workers=0, augment=True  # 0 for Windows
    )
    
    # Create model with optimal hyperparameters
    model = BandSplitRoFormer(
        num_sources=4,
        d_model=384,
        num_layers=12,
        num_heads=8,
        dim_feedforward=1536,
        dropout=0.1,
        n_fft=4096,
        sample_rate=44100
    )
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
    
    # Training configuration
    config = {
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        'num_epochs': num_epochs,
        'warmup_steps': 4000,
        'grad_clip': 1.0,
        'use_amp': True,
        'n_fft': 4096,
        'hop_length': 1024,
        'checkpoint_dir': '../checkpoints/bsroformer_full'
    }
    
    # Create trainer
    trainer = BSRoFormerTrainer(model, train_loader, val_loader, config)
    
    # Train
    trainer.train(num_epochs=num_epochs)
    
    print("\n✓ BSRoFormer training completed!")

    bsroformer_trainer = train_bsroformer_full(num_epochs=300, batch_size=8)

    return trainer# Uncomment to run full training

Starting Full BSRoFormer Training
Epochs: 300
Batch size: 8
Expected duration: 5-7 days

Found 188 files in train set
Training samples: 150
Validation samples: 38
Training batches: 18
Validation batches: 5
Model parameters: 64.89M
Model parameters: 64.89M
Starting BSRoFormer training for 300 epochs...
Starting BSRoFormer training for 300 epochs...


Validation: 100%|██████████| 5/5 [00:07<00:00,  1.45s/it]



Epoch 5:
  Train Loss: 0.3201 (mask: 0.2696, recon: 0.0505)
  Val Loss: 0.3242 (mask: 0.2739, recon: 0.0503)
Saved best checkpoint: ..\checkpoints\bsroformer_full\bsroformer_best.pt
Saved best checkpoint: ..\checkpoints\bsroformer_full\bsroformer_best.pt


Validation: 100%|██████████| 5/5 [00:07<00:00,  1.41s/it]



Epoch 10:
  Train Loss: 0.3200 (mask: 0.2696, recon: 0.0505)
  Val Loss: 0.3248 (mask: 0.2748, recon: 0.0499)
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_010.pt
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_010.pt


...

Epoch 290:
  Train Loss: 0.2346 (mask: 0.1897, recon: 0.0450)
  Val Loss: 0.2847 (mask: 0.2370, recon: 0.0477)
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_290.pt
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_290.pt


Validation: 100%|██████████| 5/5 [00:04<00:00,  1.10it/s]


Epoch 295:
  Train Loss: 0.2353 (mask: 0.1902, recon: 0.0451)
  Val Loss: 0.2834 (mask: 0.2358, recon: 0.0475)


Validation: 100%|██████████| 5/5 [00:04<00:00,  1.13it/s]



Epoch 300:
  Train Loss: 0.2355 (mask: 0.1906, recon: 0.0450)
  Val Loss: 0.2833 (mask: 0.2357, recon: 0.0476)
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_300.pt
Training completed!
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_300.pt
Training completed!
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_300.pt

✓ BSRoFormer training completed!
Saved checkpoint: ..\checkpoints\bsroformer_full\bsroformer_epoch_300.pt

✓ BSRoFormer training completed!


### Hyperparameter Optimization First, Then Full Training

In [21]:
# # Step 1: Run hyperparameter optimization for HT-Demucs
# # Uncomment to run (takes 2-3 days for 50 trials)
# best_htdemucs_params = optimize_htdemucs(
#     CONFIG['data_dir'], 
#     n_trials=50, 
#     num_epochs=20
# )

# # Step 2: Run hyperparameter optimization for BSRoFormer
# # Uncomment to run (takes 1-2 days for 50 trials)
# best_bsroformer_params = optimize_bsroformer(
#     CONFIG['data_dir'],
#     n_trials=50,
#     num_epochs=20
# )

# print("Hyperparameter optimization functions ready!")
# print("Uncomment the code above to run optimization.")